# Training notebook

## Pre-processing

In [ ]:
from multipride.preprocessing import load_tokenizer, tokenize_batch
from multipride.evaluation import compute_metrics
from multipride.data_utils import load_split
from multipride.training_utils import run_hyperparameter_search, train_save_best_model
from transformers import (
    AutoModelForSequenceClassification,
)
import torch
import wandb
import json
import os
from dotenv import load_dotenv

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0))
print("Current device:", torch.cuda.current_device())

# quick tensor test
x = torch.rand(3, 3).to("cuda")
print("Tensor on GPU:", x.device)

In [ ]:
WANDB_PROJECT_NAME = "phase_1_base"
MODEL_CHECKPOINT = "dbmdz/electra-base-italian-xxl-cased-discriminator"
MODEL_NAME = "electra-base-italian-xxl-cased"

In [ ]:
# load, and split the dataset
data_dict = load_split(
    file_format="csv",
    file_path="../data/processed/clean_data.csv",
    split=True,
    test=False,
    columns=["id", "text", "label"],
)
print(data_dict)

In [ ]:
tokenizer = load_tokenizer(model_name=MODEL_CHECKPOINT)

tokenized_train = data_dict["train"].map(
    lambda batch: tokenize_batch(batch, tokenizer), batched=True
)

tokenized_val = data_dict["test"].map(
    lambda batch: tokenize_batch(batch, tokenizer), batched=True
)

In [ ]:
print(tokenized_train)

print(tokenized_val)

In [ ]:
train_set = tokenized_train
val_set = tokenized_val

print(train_set["label"][:10])
print(val_set["label"][:10])

In [ ]:
load_dotenv("../.env")

In [ ]:
wandb.login()

In [ ]:
def model_init(trial):
    # Ensure a fresh model is loaded for each trial
    return AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)

In [ ]:
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [16, 32]
        ),
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1, log=True),
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
    }

In [ ]:
def compute_objective(metrics):
    """Returns the objective metric to maximize (Macro F1)."""
    val = metrics.get("f1") or metrics.get("eval_f1")
    if val is None:
        raise KeyError(
            f"Objective metric 'f1' not found in metrics: {list(metrics.keys())}"
        )
    return val

In [ ]:
base_path = "../artifacts"
strategy = "base"
save_path = os.path.join(base_path, WANDB_PROJECT_NAME, strategy, MODEL_NAME)
print(save_path)

os.makedirs(save_path, exist_ok=True)

In [ ]:
best_run_info, best_trainer = run_hyperparameter_search(
    model_name=MODEL_NAME,
    project_name=WANDB_PROJECT_NAME,
    train_dataset=train_set,
    eval_dataset=val_set,
    tokenizer=tokenizer,
    hp_space=optuna_hp_space,
    compute_metrics=compute_metrics,
    model_init=model_init,
    compute_objective=compute_objective,
    n_trials=3,
    output_dir=os.path.join(save_path, "hps_results"),
    logging_dir=os.path.join(save_path, "hps_logs"),
)

In [ ]:
train_save_best_model(
    best_trainer=best_trainer,
    save_path=os.path.join(save_path, "best_model"),
)

In [ ]:
with open(
    os.path.join(save_path, "best_run.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(best_run_info.hyperparameters, f, indent=4)